# 2. Multi-Agent Customer Support: Tools, Guardrails and Memory

**Learning objectives**
- Build a support agent and a quality-review agent.
- Give an agent a tool to retrieve approved information.
- Validate an output with a task guardrail.
- Understand the purpose and configuration considerations of memory.

This notebook uses a small local FAQ tool so the exercise does not depend on a third-party search API.

# Setup: CrewAI + Groq (Colab)

**Before running:** add a Groq API key to Google Colab → Secrets with the name `GROQ_API_KEY` and enable notebook access to it. No other API key is needed: this notebook uses a small local FAQ tool.

These notebooks use the current CrewAI Python API style with Groq's `openai/gpt-oss-120b` model (called through LiteLLM). The exact package versions are intentionally not pinned; after a successful lab setup, freeze the working environment for repeatability.

In [ ]:
%pip -q install -U "crewai[litellm]" crewai-tools groq

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
bigframes 2.51.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


In [ ]:
import os
from google.colab import userdata
from crewai import Agent, Task, Crew, Process, LLM

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("Add GROQ_API_KEY in Colab Secrets and enable notebook access.")

llm = LLM(
    model="groq/openai/gpt-oss-120b",
    api_key=GROQ_API_KEY,
    temperature=0.2
)
print("LLM configured (Groq).")

LLM configured (Groq).


## Step 1 — Create a small knowledge source and a custom tool

This is a deliberately small teaching knowledge base. In a real system, replace it with approved documentation and retrieval with citations.

In [ ]:
from pydantic import BaseModel, Field
from crewai.tools import BaseTool

FAQ = {
    "memory": (
        "CrewAI memory can be enabled at the crew level. Memory may retain "
        "useful information across task executions depending on the configured "
        "storage and embedding provider. Review the current CrewAI memory docs "
        "and configure the provider explicitly for production."
    ),
    "sequential": (
        "A sequential crew executes tasks in the order they are listed. "
        "The output of earlier tasks can be passed as context to later tasks."
    ),
    "tools": (
        "Tools are callable capabilities that let an agent interact with "
        "external systems or perform operations beyond text generation."
    )
}

class FAQInput(BaseModel):
    query: str = Field(..., description="The topic or question to look up in the FAQ.")

class CollegeSupportFAQTool(BaseTool):
    name: str = "Approved Support FAQ"
    description: str = (
        "Search a small approved FAQ about CrewAI concepts. "
        "Use it before answering questions about memory, sequential tasks or tools."
    )
    args_schema: type[BaseModel] = FAQInput

    def _run(self, query: str) -> str:
        q = query.lower()
        matches = [value for key, value in FAQ.items() if key in q]
        if matches:
            return "\n".join(matches)
        return (
            "No matching FAQ entry was found. State that the local knowledge "
            "base does not contain the answer; do not invent a policy."
        )

faq_tool = CollegeSupportFAQTool()
print(faq_tool.run(query="How does memory work?"))

CrewAI memory can be enabled at the crew level. Memory may retain useful information across task executions depending on the configured storage and embedding provider. Review the current CrewAI memory docs and configure the provider explicitly for production.


## Step 2 — Define the agents

The support agent has access to the FAQ tool. The QA agent checks the response and can request corrections through its own task instructions.

In [ ]:
support_agent = Agent(
    role="Customer Support Specialist",
    goal="Answer the customer's question accurately using the approved FAQ",
    backstory=(
        "You are a patient support specialist. Use the approved FAQ tool "
        "for product guidance. If the answer is not in the FAQ, say so clearly "
        "and suggest consulting official documentation."
    ),
    tools=[faq_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=5
)

qa_agent = Agent(
    role="Support Quality Reviewer",
    goal="Check that the proposed answer is accurate, complete and transparent",
    backstory=(
        "You review support answers. You remove unsupported claims, check "
        "that uncertainty is stated and ensure that all parts of the question "
        "are addressed."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=4
)

## Step 3 — Create tasks and a guardrail

A guardrail is a validation function. When it rejects an output, CrewAI can send the reason back to the agent for a retry.

In [ ]:
support_task = Task(
    description=(
        "Customer: {customer}\nContact: {person}\nQuestion: {inquiry}\n\n"
        "Use the Approved Support FAQ tool. Answer only what is supported. "
        "If the FAQ does not contain the answer, say what is missing."
    ),
    expected_output="A concise, friendly answer with a note about the information source or limitation.",
    agent=support_agent
)

def support_answer_guardrail(output):
    text = output.raw.strip()
    if len(text) < 40:
        return (False, "The answer is too brief. Address the customer's question in complete sentences.")
    if "invent" in text.lower():
        return (False, "Rewrite the answer without unsupported assumptions.")
    return (True, text)

qa_task = Task(
    description=(
        "Review the draft response for {customer}. Confirm that it answers "
        "the question, uses only the available FAQ information, and clearly "
        "states any limitation. Return the corrected final answer."
    ),
    expected_output="A final customer-ready answer, with unsupported claims removed.",
    agent=qa_agent,
    guardrail=support_answer_guardrail,
    guardrail_max_retries=2
)

## Step 4 — Assemble and run the Crew

In [ ]:
support_crew = Crew(
    agents=[support_agent, qa_agent],
    tasks=[support_task, qa_task],
    process=Process.sequential,
    verbose=True
)

inputs = {
    "customer": "CampusHelp (fictional customer)",
    "person": "Aarav",
    "inquiry": "How can I enable memory for a CrewAI crew, and what should I consider?"
}

support_result = await support_crew.kickoff_async(inputs=inputs)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6883de22-88a3-47af-8b2c-fb9441d9fb74                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Customer: CampusHelp (fictional customer)                                                                │
│  Contact: Aarav                                                                                                 │
│  Question: How can I enable memory for a CrewAI crew, and what should I consider?                               │
│                                                                                                                 │
│  Use the Approved Support FAQ tool. Answer only what is supported. If the FAQ does not contain the answer, say  │
│  what is missing.                                                                                               │
│  ID: bd948a56-e559-47f1-84c2-6fc067d0420d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Specialist                                                                             │
│                                                                                                                 │
│  Task: Customer: CampusHelp (fictional customer)                                                                │
│  Contact: Aarav                                                                                                 │
│  Question: How can I enable memory for a CrewAI crew, and what should I consider?                               │
│                                                                                                                 │
│  Use the Approved Support FAQ tool. Answer only what is supported. If the FAQ does not contain the answer, say  │
│  what is missing.                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: approved_support_faq                                                                                     │
│  Args: {'query': 'enable memory for a CrewAI crew considerations'}                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#1) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: approved_support_faq                                                                                     │
│  Output: CrewAI memory can be enabled at the crew level. Memory may retain useful information across task       │
│  executions depending on the configured storage and embedding provider. Review the current CrewAI memory docs   │
│  and configure the provider explicitly for production.                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


... (6 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the draft response for CampusHelp (fictional customer). Confirm that it answers the question,     │
│  uses only the available FAQ information, and clearly states any limitation. Return the corrected final         │
│  answer.                                                                                                        │
│  Agent: Support Quality Reviewer                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── 🛡️ Guardrail Success ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Passed                                                                                               │
│  Name: Validation Successful                                                                                    │
│  Status: ✅ Validated                                                                                           │
│  Attempts: 1                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🛡️ Guardrail Check ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Evaluation Started                                                                                   │
│  Name: def support_answer_guardrail(output):                                                                    │
│      text = o...                                                                                                │
│  Status: 🔄 Evaluating                                                                                          │
│  Attempt: 1                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 6883de22-88a3-47af-8b2c-fb9441d9fb74                                                                       │
│  Final Output: **Answer**                                                                                       │
│                                                                                                                 │
│  You can enable CrewAI’s memory at the **crew level**. Once memory is turned on, the crew can keep useful       │
│  information between task executions, but the actual retention depends on the **storage and embedding           │
│  provider** you configure.                                                                                      │
│                                                                                                                 │
│  **For a production setup you should:**                                                                         │
│                                                                                                                 │
│  1. **Explicitly configure a persistent storage provider** (e.g., a vector database such as Pinecone,           │
│  Weaviate, or any other supported persistent store).                                                            │
│  2. **Select an embedding provider** that matches your production requirements (e.g., OpenAI embeddings,        │
│  Cohere, etc.).                                                                                                 │
│  3. **Set these providers in the crew configuration** so that memory is written to and read from the            │
│  persistent store rather than an in‑memory fallback.                                                            │
│                                                                                                                 │
│  **Why this is necessary**                                                                                      │
│                                                                                                                 │
│  - If you rely on the default (often in‑memory) storage, the memory will be lost when the process restarts.     │
│  - The quality and durability of the retained information are directly tied to the capabilities of the chosen   │
│  storage and embedding services.                                                                                │
│                                                                                                                 │
│  **Limitations / Things to keep in mind**                                                                       │
│                                                                                                                 │
│  - The FAQ only states that memory “may retain useful information” – it does **not guarantee** that all data    │
│  will be persisted indefinitely; retention is subject to the behavior of the configured providers.              │
│  - Some embedding providers have rate limits or token quotas that could affect how much data you can store.     │
│  - If you need specific retention policies (e.g., expiration, encryption), you’ll need to verify that your      │
│  chosen storage solution supports those features, as the FAQ does not cover them.                               │
│                                                                                                                 │
│  **Next steps**                                       

In [ ]:
from IPython.display import Markdown, display
display(Markdown(support_result.raw))

**Answer**

You can enable CrewAI’s memory at the **crew level**. Once memory is turned on, the crew can keep useful information between task executions, but the actual retention depends on the **storage and embedding provider** you configure.

**For a production setup you should:**

1. **Explicitly configure a persistent storage provider** (e.g., a vector database such as Pinecone, Weaviate, or any other supported persistent store).  
2. **Select an embedding provider** that matches your production requirements (e.g., OpenAI embeddings, Cohere, etc.).  
3. **Set these providers in the crew configuration** so that memory is written to and read from the persistent store rather than an in‑memory fallback.

**Why this is necessary**

- If you rely on the default (often in‑memory) storage, the memory will be lost when the process restarts.  
- The quality and durability of the retained information are directly tied to the capabilities of the chosen storage and embedding services.

**Limitations / Things to keep in mind**

- The FAQ only states that memory “may retain useful information” – it does **not guarantee** that all data will be persisted indefinitely; retention is subject to the behavior of the configured providers.  
- Some embedding providers have rate limits or token quotas that could affect how much data you can store.  
- If you need specific retention policies (e.g., expiration, encryption), you’ll need to verify that your chosen storage solution supports those features, as the FAQ does not cover them.

**Next steps**

- Review the current CrewAI memory documentation for the exact configuration syntax.  
- Choose a production‑grade storage and embedding provider that meets your reliability and compliance needs.  
- Update your crew definition to include the explicit provider settings, then test that memory persists across multiple task runs in your production environment.  

If you run into any issues while configuring the providers, feel free to share the relevant configuration snippet and we can help troubleshoot further.

## Step 5 — Memory: understand before enabling

CrewAI memory is not the same as simply keeping the current prompt in context. A memory-enabled crew can store and retrieve information across runs, depending on its configuration.

The following is an **optional demonstration**. Run it after confirming the embedding provider and storage configuration for your environment. Memory may make additional model/API calls and can persist data locally.

> **Groq note:** Groq provides chat models but not an embedding model, so memory needs an explicitly configured embedder (for example a local or third-party embedding provider). The cell below only *creates* the crew inside a `try/except`; the runs stay commented out and are **not** required for the exercises.


In [ ]:
# OPTIONAL: Enable built-in CrewAI memory.
# Groq has no embedding model, so configure an embedder explicitly before using memory.
# Check the current CrewAI Memory documentation.

try:
    memory_demo_crew = Crew(
        agents=[support_agent],
        tasks=[Task(
            description=(
                "The customer {customer} says: {message}. "
                "Respond briefly and remember only the non-sensitive preference "
                "that the customer has explicitly provided."
            ),
            expected_output="A short acknowledgement.",
            agent=support_agent
        )],
        process=Process.sequential,
        memory=True,
        verbose=True
    )
    print("Memory demo crew created (not executed).")
except Exception as e:
    print("Memory demo skipped:", str(e)[:200])

# First run: state a harmless preference.
# Uncomment to execute (only after configuring an embedder):
# memory_demo_crew.kickoff(inputs={
#     "customer": "CampusHelp",
#     "message": "For future answers, please use concise bullet points."
# })

# Second run: ask whether the preference can be applied.
# Uncomment to execute after the first run:
# memory_demo_crew.kickoff(inputs={
#     "customer": "CampusHelp",
#     "message": "Explain sequential task execution."
# })

Memory demo crew created (not executed).


## Try it yourself
1. Ask about `sequential`, `tools` and an unknown topic.
2. Modify the FAQ tool to add a new approved entry.
3. Deliberately make the guardrail fail and observe the retry.
4. Compare the answer with and without the QA task.

**Safety note:** Never store passwords, API keys, payment details or sensitive personal information in agent memory. Explain what is retained and provide a way to clear it.

# Exercise solutions — Try it yourself

All cells below only **use** objects created in the cells above (`llm`, `faq_tool`, `FAQ`, `support_agent`, `qa_agent`, `support_answer_guardrail`). No original cell is modified.

## Exercise setup — helper

`ask(...)` builds **fresh** tasks (same text as Step 3) and runs a crew, then records time and length in `runs`. Fresh tasks are used because `support_task`/`qa_task` above were already run and hold their previous output.

In [ ]:
import time
import pandas as pd
from IPython.display import Markdown, display

runs = []   # one row per run, used in the results summary

async def ask(inquiry, label, use_qa=True, guardrail_fn=support_answer_guardrail, retries=2,
              customer="CampusHelp (fictional customer)", person="Aarav"):
    # same wording as support_task in Step 3
    s_task = Task(
        description=(
            "Customer: {customer}\nContact: {person}\nQuestion: {inquiry}\n\n"
            "Use the Approved Support FAQ tool. Answer only what is supported. "
            "If the FAQ does not contain the answer, say what is missing."
        ),
        expected_output="A concise, friendly answer with a note about the information source or limitation.",
        agent=support_agent
    )
    agents, tasks = [support_agent], [s_task]

    if use_qa:
        # same wording as qa_task in Step 3, but the guardrail is configurable
        q_task = Task(
            description=(
                "Review the draft response for {customer}. Confirm that it answers "
                "the question, uses only the available FAQ information, and clearly "
                "states any limitation. Return the corrected final answer."
            ),
            expected_output="A final customer-ready answer, with unsupported claims removed.",
            agent=qa_agent,
            guardrail=guardrail_fn,
            guardrail_max_retries=retries
        )
        agents.append(qa_agent)
        tasks.append(q_task)

    crew = Crew(agents=agents, tasks=tasks, process=Process.sequential, verbose=False)

    start = time.time()
    result = await crew.kickoff_async(
        inputs={"customer": customer, "person": person, "inquiry": inquiry}
    )
    elapsed = round(time.time() - start, 1)

    ok, _ = support_answer_guardrail(result)
    runs.append({"run": label, "qa_task": use_qa, "seconds": elapsed,
                 "words": len(result.raw.split()), "std_guardrail_ok": ok})
    return result, elapsed

print("Helper ready.")

Helper ready.


## Exercise 1 — Ask about `sequential`, `tools` and an unknown topic

The FAQ covers the first two. The third is **not** in the FAQ, so a correct answer must say the information is missing instead of inventing a refund policy.

In [ ]:
questions = [
    ("Ex1: sequential", "What is a sequential crew and how does it execute tasks?"),
    ("Ex1: tools",      "What are tools in CrewAI and why would an agent need them?"),
    ("Ex1: UNKNOWN",    "What is the refund policy for CampusHelp premium plans?"),
]

for label, q in questions:
    print("=" * 90)
    print("QUESTION:", q)
    res, secs = await ask(q, label=label)
    display(Markdown(res.raw))
    print(f"({secs} s)")

QUESTION: What is a sequential crew and how does it execute tasks?


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Specialist                                                                             │
│                                                                                                                 │
│  Task: Customer: CampusHelp (fictional customer)                                                                │
│  Contact: Aarav                                                                                                 │
│  Question: What is a sequential crew and how does it execute tasks?                                             │
│                                                                                                                 │
│  Use the Approved Support FAQ tool. Answer only what is supported. If the FAQ does not contain the answer, say  │
│  what is missing.                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Specialist                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Hi Aarav! A sequential crew executes tasks in the order they are listed. The output of earlier tasks can be    │
│  passed as context to later tasks.                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Support Quality Reviewer                                                                                │
│                                                                                                                 │
│  Task: Review the draft response for CampusHelp (fictional customer). Confirm that it answers the question,     │
│  uses only the available FAQ information, and clearly states any limitation. Return the corrected final         │
│  answer.                                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


... (15 outputs left out)

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Support Quality Reviewer                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  I’m sorry, but the approved FAQ does not include information about the refund policy for CampusHelp premium    │
│  plans. For the most accurate and up‑to‑date details, please refer to the official CampusHelp documentation or  │
│  contact their support team directly.                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

I’m sorry, but the approved FAQ does not include information about the refund policy for CampusHelp premium plans. For the most accurate and up‑to‑date details, please refer to the official CampusHelp documentation or contact their support team directly.

(1.4 s)


## Exercise 2 — Modify the FAQ tool: add new approved entries

`CollegeSupportFAQTool._run` reads the global `FAQ` dictionary every time it is called, so adding keys is enough. An entry is matched when its **key appears in the question text** (lower-case).

In [ ]:
FAQ["guardrail"] = (
    "A guardrail is a validation function attached to a task. It checks the task "
    "output; if it returns (False, reason), CrewAI sends the reason back to the agent "
    "so it can retry, up to guardrail_max_retries times."
)
FAQ["delegation"] = (
    "Delegation lets an agent hand part of its work to another agent. It is "
    "controlled by allow_delegation on the agent and is disabled in this lab."
)

# 1) test the tool directly
print(faq_tool.run(query="How does a guardrail work?"))
print("-" * 60)
print(faq_tool.run(query="Can agents use delegation?"))
print("=" * 90)

# 2) test through the crew
res, secs = await ask(
    "How does a guardrail work in CrewAI, and how many retries can it do?",
    label="Ex2: new FAQ entry (guardrail)"
)
display(Markdown(res.raw))
print(f"({secs} s)")

A guardrail is a validation function attached to a task. It checks the task output; if it returns (False, reason), CrewAI sends the reason back to the agent so it can retry, up to guardrail_max_retries times.
------------------------------------------------------------
Delegation lets an agent hand part of its work to another agent. It is controlled by allow_delegation on the agent and is disabled in this lab.


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Specialist                                                                             │
│                                                                                                                 │
│  Task: Customer: CampusHelp (fictional customer)                                                                │
│  Contact: Aarav                                                                                                 │
│  Question: How does a guardrail work in CrewAI, and how many retries can it do?                                 │
│                                                                                                                 │
│  Use the Approved Support FAQ tool. Answer only what is supported. If the FAQ does not contain the answer, say  │
│  what is missing.                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Specialist                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  A guardrail is a validation function attached to a task. It checks the task output; if it returns **(False,    │
│  reason)**, CrewAI sends the reason back to the agent so it can retry, up to **guardrail_max_retries** times.   │
│                                                                                                                 │
│  *Information sourced from the approved CrewAI support FAQ.*                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Support Quality Reviewer                                                                                │
│                                                                                                                 │
│  Task: Review the draft response for CampusHelp (fictional customer). Confirm that it answers the question,     │
│  uses only the available FAQ information, and clearly states any limitation. Return the corrected final         │
│  answer.                                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Support Quality Reviewer                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  A **guardrail** in CrewAI is a validation function that is attached to a task.                                 │
│  When the task finishes, the guardrail runs and evaluates the output:                                           │
│                                                                                                                 │
│  * If the guardrail returns **(True, …)**, the output is accepted and the workflow continues.                   │
│  * If it returns **(False, reason)**, CrewAI sends that reason back to the agent so it can retry the task.      │
│                                                                                                                 │
│  The agent will automatically retry the task up to the number of times specified by                             │
│  **`guardrail_max_retries`**. After the maximum number of retries is reached, the workflow proceeds with the    │
│  last result (or fails, depending on how the task is configured).                                               │
│                                                                                                                 │
│  **Limitations**                                                                                                │
│  * The behavior described is limited to what is documented in the CrewAI support FAQ.                           │
│  * Guardrails only affect the specific task they are attached to; they do not enforce constraints across        │
│  unrelated tasks.                                                                                               │
│  * The number of retries is fixed by the `guardrail_max_retries` setting—if you need more attempts, the         │
│  setting must be increased in the task configuration.                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

A **guardrail** in CrewAI is a validation function that is attached to a task.  
When the task finishes, the guardrail runs and evaluates the output:

* If the guardrail returns **(True, …)**, the output is accepted and the workflow continues.  
* If it returns **(False, reason)**, CrewAI sends that reason back to the agent so it can retry the task.

The agent will automatically retry the task up to the number of times specified by **`guardrail_max_retries`**. After the maximum number of retries is reached, the workflow proceeds with the last result (or fails, depending on how the task is configured).

**Limitations**  
* The behavior described is limited to what is documented in the CrewAI support FAQ.  
* Guardrails only affect the specific task they are attached to; they do not enforce constraints across unrelated tasks.  
* The number of retries is fixed by the `guardrail_max_retries` setting—if you need more attempts, the setting must be increased in the task configuration.

(2.4 s)


## Exercise 3 — Deliberately make the guardrail fail and observe the retry

**Part A:** a guardrail that fails on the first attempt (with feedback) and then checks that the agent followed the feedback. The printed log shows each attempt.

**Part B:** an impossible guardrail. After the allowed retries CrewAI raises an exception, which shows what happens when the feedback can never be satisfied.

In [ ]:
# ---------- Part A: fail once, then recover ----------
state = {"attempts": 0}

def fail_once_guardrail(output):
    state["attempts"] += 1
    n = state["attempts"]
    text = output.raw.strip()
    if n == 1:
        print(f"[guardrail] attempt {n}: FAIL (deliberate)")
        return (False, "Deliberate test failure. Rewrite the answer so that it starts exactly with 'Hello Aarav,'.")
    ok = text.startswith("Hello Aarav")
    print(f"[guardrail] attempt {n}: {'PASS' if ok else 'FAIL'}")
    if not ok:
        return (False, "The answer must start exactly with 'Hello Aarav,'.")
    return (True, text)

res_a, secs = await ask(
    "Explain sequential task execution in two sentences.",
    label="Ex3A: fail-once guardrail",
    guardrail_fn=fail_once_guardrail,
    retries=3,
)
print(f"\nGuardrail was called {state['attempts']} time(s).")
display(Markdown(res_a.raw))

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Specialist                                                                             │
│                                                                                                                 │
│  Task: Customer: CampusHelp (fictional customer)                                                                │
│  Contact: Aarav                                                                                                 │
│  Question: Explain sequential task execution in two sentences.                                                  │
│                                                                                                                 │
│  Use the Approved Support FAQ tool. Answer only what is supported. If the FAQ does not contain the answer, say  │
│  what is missing.                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Specialist                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  A sequential crew executes tasks in the order they are listed. The output of earlier tasks can be passed as    │
│  context to later tasks. (Based on the approved support FAQ)                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Support Quality Reviewer                                                                                │
│                                                                                                                 │
│  Task: Review the draft response for CampusHelp (fictional customer). Confirm that it answers the question,     │
│  uses only the available FAQ information, and clearly states any limitation. Return the corrected final         │
│  answer.                                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Support Quality Reviewer                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  I’m sorry, but I can’t complete the review without seeing the draft response (and the original question) that  │
│  you’d like me to evaluate. Could you please provide the draft answer you’d like reviewed? Once I have that, I  │
│  can check it for accuracy, completeness, and proper use of the FAQ information, and then supply a corrected,   │
│  customer‑ready version.                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (2 outputs left out)

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Support Quality Reviewer                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Hello Aarav,                                                                                                   │
│                                                                                                                 │
│  Thank you for reaching out. To ensure I can review the draft response accurately and confirm that it aligns    │
│  with our FAQ guidelines, could you please provide the draft answer you’d like evaluated (as well as the        │
│  original customer question, if possible)? Once I have that information, I’ll check it for accuracy,            │
│  completeness, and proper use of the available FAQ content, and then supply a corrected, customer‑ready         │
│  version.                                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[guardrail] attempt 2: PASS

Guardrail was called 2 time(s).


Hello Aarav,

Thank you for reaching out. To ensure I can review the draft response accurately and confirm that it aligns with our FAQ guidelines, could you please provide the draft answer you’d like evaluated (as well as the original customer question, if possible)? Once I have that information, I’ll check it for accuracy, completeness, and proper use of the available FAQ content, and then supply a corrected, customer‑ready version.

In [ ]:
# ---------- Part B: impossible guardrail -> retries are exhausted ----------
state_b = {"attempts": 0}

def impossible_guardrail(output):
    state_b["attempts"] += 1
    print(f"[guardrail] attempt {state_b['attempts']}: FAIL (impossible rule)")
    return (False, "The answer must contain the exact token XYZ-IMPOSSIBLE-123.")

try:
    await ask("What are tools in CrewAI?", label="Ex3B: impossible guardrail",
              guardrail_fn=impossible_guardrail, retries=2)
    print("No exception was raised (this CrewAI version may not enforce the retry limit).")
except Exception as e:
    print(f"\nTask failed after {state_b['attempts']} guardrail attempt(s) "
          f"(1 initial + up to 2 retries).")
    print("Error type:", type(e).__name__)
    print("Message   :", str(e)[:300])

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Specialist                                                                             │
│                                                                                                                 │
│  Task: Customer: CampusHelp (fictional customer)                                                                │
│  Contact: Aarav                                                                                                 │
│  Question: What are tools in CrewAI?                                                                            │
│                                                                                                                 │
│  Use the Approved Support FAQ tool. Answer only what is supported. If the FAQ does not contain the answer, say  │
│  what is missing.                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Specialist                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Tools are callable capabilities that let an agent interact with external systems or perform operations beyond  │
│  text generation. (Answer sourced from the approved CrewAI FAQ)                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Support Quality Reviewer                                                                                │
│                                                                                                                 │
│  Task: Review the draft response for CampusHelp (fictional customer). Confirm that it answers the question,     │
│  uses only the available FAQ information, and clearly states any limitation. Return the corrected final         │
│  answer.                                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Support Quality Reviewer                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  I’m unable to complete the review because the draft response and the original customer question are not        │
│  provided. To produce a corrected, customer‑ready answer I would need to see both the question you’re           │
│  responding to and the draft answer you’d like me to review. Please share those details, and I’ll gladly check  │
│  the response for accuracy, completeness, and proper use of the FAQ information.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (5 outputs left out)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Support Quality Reviewer                                                                                │
│                                                                                                                 │
│  Task: Review the draft response for CampusHelp (fictional customer). Confirm that it answers the question,     │
│  uses only the available FAQ information, and clearly states any limitation. Return the corrected final         │
│  answer.                                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Support Quality Reviewer                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  XYZ-IMPOSSIBLE-123                                                                                             │
│                                                                                                                 │
│  I’m unable to complete the review because the draft response and the original customer question have not been  │
│  provided. To produce a corrected, customer‑ready answer I would need to see both the question you’re           │
│  responding to and the draft answer you’d like me to review. Please share those details, and I’ll gladly check  │
│  the response for accuracy, completeness, and proper use of the FAQ information.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[guardrail] attempt 3: FAIL (impossible rule)

Task failed after 3 guardrail attempt(s) (1 initial + up to 2 retries).
Error type: Exception
Message   : Task failed guardrail validation after 2 retries. Last error: The answer must contain the exact token XYZ-IMPOSSIBLE-123.


## Exercise 4 — Compare the answer with and without the QA task

Same question, same support agent. One crew has only the support task; the other adds the QA task (with the standard guardrail from Step 3).

In [ ]:
Q = "How can I enable memory for a CrewAI crew, and what should I consider?"

res_no_qa, t_no = await ask(Q, label="Ex4: WITHOUT QA task", use_qa=False)
res_qa,    t_qa = await ask(Q, label="Ex4: WITH QA task",    use_qa=True)

display(Markdown("### Without QA task\n\n" + res_no_qa.raw))
display(Markdown("---\n### With QA task\n\n" + res_qa.raw))

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Specialist                                                                             │
│                                                                                                                 │
│  Task: Customer: CampusHelp (fictional customer)                                                                │
│  Contact: Aarav                                                                                                 │
│  Question: How can I enable memory for a CrewAI crew, and what should I consider?                               │
│                                                                                                                 │
│  Use the Approved Support FAQ tool. Answer only what is supported. If the FAQ does not contain the answer, say  │
│  what is missing.                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Specialist                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  According to our support FAQ:                                                                                  │
│                                                                                                                 │
│  **CrewAI memory can be enabled at the crew level. Memory may retain useful information across task executions  │
│  depending on the configured storage and embedding provider. Review the current CrewAI memory docs and          │
│  configure the provider explicitly for production.**                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Specialist                                                                             │
│                                                                                                                 │
│  Task: Customer: CampusHelp (fictional customer)                                                                │
│  Contact: Aarav                                                                                                 │
│  Question: How can I enable memory for a CrewAI crew, and what should I consider?                               │
│                                                                                                                 │
│  Use the Approved Support FAQ tool. Answer only what is supported. If the FAQ does not contain the answer, say  │
│  what is missing.                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Specialist                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  CrewAI memory can be enabled at the crew level. Memory may retain useful information across task executions    │
│  depending on the configured storage and embedding provider. Review the current CrewAI memory docs and          │
│  configure the provider explicitly for production.                                                              │
│                                                                                                                 │
│  *Note: This information is taken directly from the approved Support FAQ.*                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Support Quality Reviewer                                                                                │
│                                                                                                                 │
│  Task: Review the draft response for CampusHelp (fictional customer). Confirm that it answers the question,     │
│  uses only the available FAQ information, and clearly states any limitation. Return the corrected final         │
│  answer.                                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Support Quality Reviewer                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Answer**                                                                                                     │
│                                                                                                                 │
│  You can enable CrewAI memory at the **crew level**. When memory is turned on, it can retain useful             │
│  information across task executions, but the actual persistence depends on the **storage and embedding          │
│  provider** you configure.                                                                                      │
│                                                                                                                 │
│  **Steps for production use**                                                                                   │
│                                                                                                                 │
│  1. **Enable memory for the crew** – set the `memory` flag (or equivalent setting) in your crew configuration.  │
│  2. **Choose a storage provider** – select a persistent storage backend (e.g., a database, Redis, or a          │
│  cloud‑based vector store) that will keep the memory data between runs.                                         │
│  3. **Select an embedding provider** – configure an embedding model/provider that will be used to encode the    │
│  stored information.                                                                                            │
│  4. **Configure the provider explicitly** – in your production configuration file or environment, specify both  │
│  the storage and embedding providers so that they are not left to default or temporary settings.                │
│                                                                                                                 │
│  **Important limitation**                                                                                       │
│                                                                                                                 │
│  - The retention of information is **only as reliable as the configured providers**. If you use an in‑memory    │
│  or temporary storage option, the memory will be lost when the process restarts. Likewise, if the embedding     │
│  provider is not set to a production‑grade service, the quality of stored context may be reduced.               │
│                                                                                                                 │
│  **Summary**                                                                                                    │
│                                                                                                                 │
│  - Enable memory at the crew level.                                                                             │
│  - Explicitly configure a persistent storage backend and a production‑grade embedding provider.                 │
│  - Be aware that memory persistence is tied to those provider choices; using non‑persistent defaults will       │
│  cause data loss.                                                                                               │
│                                                                                                                 │
│  If you need help selecting or configuring a specific s

### Without QA task

According to our support FAQ:

**CrewAI memory can be enabled at the crew level. Memory may retain useful information across task executions depending on the configured storage and embedding provider. Review the current CrewAI memory docs and configure the provider explicitly for production.**

---
### With QA task

**Answer**

You can enable CrewAI memory at the **crew level**. When memory is turned on, it can retain useful information across task executions, but the actual persistence depends on the **storage and embedding provider** you configure.

**Steps for production use**

1. **Enable memory for the crew** – set the `memory` flag (or equivalent setting) in your crew configuration.  
2. **Choose a storage provider** – select a persistent storage backend (e.g., a database, Redis, or a cloud‑based vector store) that will keep the memory data between runs.  
3. **Select an embedding provider** – configure an embedding model/provider that will be used to encode the stored information.  
4. **Configure the provider explicitly** – in your production configuration file or environment, specify both the storage and embedding providers so that they are not left to default or temporary settings.  

**Important limitation**

- The retention of information is **only as reliable as the configured providers**. If you use an in‑memory or temporary storage option, the memory will be lost when the process restarts. Likewise, if the embedding provider is not set to a production‑grade service, the quality of stored context may be reduced.

**Summary**

- Enable memory at the crew level.  
- Explicitly configure a persistent storage backend and a production‑grade embedding provider.  
- Be aware that memory persistence is tied to those provider choices; using non‑persistent defaults will cause data loss.  

If you need help selecting or configuring a specific storage or embedding service, let us know and we can provide guidance on the available options.

## Results summary

In [ ]:
summary = pd.DataFrame(runs)
display(summary)

a = next((r for r in runs if r["run"] == "Ex4: WITHOUT QA task"), None)
b = next((r for r in runs if r["run"] == "Ex4: WITH QA task"), None)
if a and b:
    print(f"\nThe QA task made the run {b['seconds']/a['seconds']:.1f}x as long "
          f"({a['seconds']} s -> {b['seconds']} s) and changed the answer length "
          f"from {a['words']} to {b['words']} words.")

,run,qa_task,seconds,words,std_guardrail_ok
0,Ex1: sequential,True,3.3,376,True
1,Ex1: tools,True,2.1,80,True
2,Ex1: UNKNOWN,True,1.4,38,True
3,Ex2: new FAQ entry (guardrail),True,2.4,159,True
4,Ex3A: fail-once guardrail,True,2.4,70,True
5,Ex4: WITHOUT QA task,False,2.1,43,True
6,Ex4: WITH QA task,True,12.7,253,True



The QA task made the run 6.0x as long (2.1 s -> 12.7 s) and changed the answer length from 43 to 253 words.


# Exercise Observations

## Exercise 1: FAQ questions
- The support agent answered the `sequential` and `tools` questions directly from the FAQ tool and noted the source.
- The QA reviewer was less reliable: for "tools" it asked for the draft instead of giving a final answer, and for "sequential" it added details not in the FAQ (placeholder syntax, "no parallel execution").
- The unknown question (CampusHelp refund policy) returned "No matching FAQ entry", and the final answer correctly said the information is not available and pointed to official documentation instead of inventing a policy.

## Exercise 2: Adding FAQ entries
- Added `guardrail` and `delegation` entries to the `FAQ` dictionary.
- The tool returned the new text when called directly, and the support agent's draft about guardrails matched it, including the `guardrail_max_retries` retry behaviour.
- The QA answer added a few claims not in the FAQ, so the QA step does not always stay within the approved information.
- Matching works only when the key word (e.g. "guardrail") appears in the question.

## Exercise 3: Guardrail retries
- Part A: the guardrail failed attempt 1 on purpose, CrewAI printed "retrying due to ...", and attempt 2 passed with an answer starting "Hello Aarav,". The guardrail was called 2 times. It passed only because of the prefix; the body still asked for the draft, so a guardrail only checks what it is coded to check.
- Part B: the impossible guardrail failed 3 times (1 initial + 2 retries) and CrewAI raised an exception: "Task failed guardrail validation after 2 retries."

## Exercise 4: With vs without QA
- Without QA: 2.1 s, 43 words, a short answer taken directly from the FAQ.
- With QA: 12.7 s, 253 words, about 6x longer.
- The QA answer was longer and more detailed but added unsupported details (e.g. Redis, vector stores) that are not in the FAQ, so it did not stay within the FAQ.
- The QA task adds another LLM call and time, and it improves reliability only if it stays restricted to the approved information.

## Conclusion
These exercises showed how a sequential crew combines a custom FAQ tool, a QA reviewer and task guardrails with retries. The FAQ tool kept the support answers grounded, and guardrails can force a retry or stop a task after the retry limit. The QA step adds time and did not always stay within the FAQ, so its prompt and guardrail need to be strict.